In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from statsmodels.nonparametric.kde import KDEUnivariate

In [ ]:
# Define reusable functions
def load_data(input_folder):
    """Load and combine CSV files from a given folder into a single DataFrame."""
    dataframes = [pd.read_csv(os.path.join(input_folder, file)) for file in os.listdir(input_folder) if file.endswith('.csv')]
    return pd.concat(dataframes, ignore_index=True)

In [ ]:
# Select key features to analyze
key_features = ['dl_bitrate', 'ul_err']

In [ ]:
# EDA for main dataset
input_folder = '../Datasets/Per_UE_Datasets_final'
combined_data = load_data(input_folder)
combined_data['dl_bitrate'] = combined_data['dl_bitrate']/(1024*1024)
normal_data, attack_data = combined_data[combined_data["binary_label"] == 0], combined_data[combined_data["binary_label"] == 1]

In [ ]:
def plot_distributions_individual(data, features, group_column, is_multiclass=False, output_folder="plots", custom_limits=None):
    """
    Plot KDE distributions and save individual subplots for each feature and label.

    Parameters:
    - data: pandas DataFrame containing the data.
    - features: List of feature names to plot.
    - group_column: The column containing the group labels (e.g., 0 for Normal, 1 for Attack).
    - is_multiclass: Whether the group_column contains multiclass labels.
    - output_folder: Directory to save the plots.
    - custom_limits: Dictionary of custom axis limits. Format:
        {
            "feature_name": {
                0: {"x": (xmin, xmax), "y": (ymin, ymax)},  # Limits for Normal
                1: {"x": (xmin, xmax), "y": (ymin, ymax)}   # Limits for Attack
            }
        }
    """
    os.makedirs(output_folder, exist_ok=True)

    for feature in features:
        for label in data[group_column].unique():
            subset = data[data[group_column] == label][feature].dropna()
            label_name = "Normal" if label == 0 else (f"Attack Type {int(label)}" if is_multiclass else "Attack")
            
            if subset.nunique() > 1:
                # Create a new figure for each subplot
                plt.figure(figsize=(3, 2.5))  # Adjust figure size for individual plots
                color = "blue" if label == 0 else "red"

                # Plot KDE
                sns.kdeplot(
                    data=subset, 
                    fill=True, 
                    alpha=0.70, 
                    bw_adjust=0.5,
                    linewidth=4,
                    color=color,
                    label=label_name
                )
                # Set title and labels
                #plt.title(f'{label_name}: {feature}', fontsize=14)
                #plt.xlabel('Downlink Bitrate (Mbps)' if "dl_bitrate" in feature else 'Uplink Error', fontsize=12)
                plt.xlabel('')
                plt.ylabel("Density", fontsize=14)
                plt.grid(True, linestyle='--', alpha=0.95)
                plt.xticks(fontsize=12)
                plt.yticks(fontsize=12)
                #plt.legend(fontsize=10)

                # Apply custom axis limits if provided
                if custom_limits and feature in custom_limits and label in custom_limits[feature]:
                    limits = custom_limits[feature][label]
                    if "x" in limits:
                        plt.xlim(limits["x"])  # Apply x-axis limits
                    if "y" in limits:
                        plt.ylim(limits["y"])  # Apply y-axis limits

                # Save each subplot as an individual file
                filename = f"{output_folder}/{feature}_{label_name.replace(' ', '_')}.png"
                plt.savefig(filename, dpi=300, bbox_inches='tight')
                print(f"Saved: {filename}")
                plt.close()
            else:
                print(f"Skipping {feature} for {label_name} due to zero variance.")


In [ ]:
custom_limits = {
    "dl_bitrate": {
        0: {"x": (0, 5), "y": (0, 0.4)},  # Normal
        1: {"x": (0, 0.25), "y": (0, 30)}  # Attack
    },
    "ul_err": {
        0: {"x": (0, 6), "y": (0, 0.3)},  # Normal
        1: {"x": (0, 20), "y": (0, 0.5)}  # Attack
    }
}

plot_distributions_individual(
    data=combined_data,
    features=['dl_bitrate', 'ul_err'],
    group_column='binary_label',
    custom_limits=custom_limits,
    output_folder="individual_plots"
)
print("done")

In [ ]:
The analysis shows a big difference between normal and attack conditions in network performance.
Uplink errors increase by over 2,900%, and downlink errors go up by 546% during attacks.
The downlink bitrate almost completely drops, decreasing by 96%. 
The graphs clearly show this: uplink errors spike, and downlink bitrate falls close to zero during attacks. 
Transmission rates and signal quality also get much worse. 
These results show that attacks heavily damage communication and data flow. 